# 6 - Spatial strategies: 5-seed robustness

Rigorous follow-up to notebook 5. Trains each spatial strategy at **5 seeds** and reports the
**seed-averaged** fake-cloud MAE/RMSE and error-over-time, so we can tell whether the single-seed
ranking (random best) survives run-to-run training noise. Same region, channels, split, and shared
stats as nb5. Maps stay single-model in nb5 (averages have no per-pixel map). patch80 is excluded
(80x112 fails cuDNN on this GPU).

In [ ]:
import os, pickle
import numpy as np, xarray as xr, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

RECHUNKED = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
ORIGINAL  = os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr")

SUBSET = dict(lat=slice(31, 5), lon=slice(42, 80))     # Arab Sea (~104x152), same as nb5
features = []
train_year, train_range, val_range, test_range = 2015, 3, 1, 1
TEST_YEAR = train_year + train_range + val_range        # 2019
DAY_BATCH = 100
SEEDS = [0, 1, 2, 3, 4]

MODEL_DIR = "models/spatial_strat"
os.makedirs(MODEL_DIR, exist_ok=True)

STRATS = {
    "whole":         dict(tile=None,      overlap=None,                 batch=1,  sampling="grid"),
    "nonoverlap":    dict(tile=(40, 56),  overlap=(0, 0),               batch=16, sampling="grid"),
    "overlap":       dict(tile=(40, 56),  overlap=(20, 28),             batch=16, sampling="grid"),
    "overlap_small": dict(tile=(40, 56),  overlap=(4, 6),               batch=16, sampling="grid"),
    "random":        dict(tile=(40, 56),  overlap=None, n_per_day=16,   batch=16, sampling="random"),
}

def open_region(path):
    ds = xr.open_zarr(path, chunks={})
    if SUBSET is not None:
        ds = ds.sel(**SUBSET)
    ds = mtg.crop_to_multiple(ds, multiple=8)
    return ds.sel(time=slice(f"{train_year}-01-01",
                             f"{train_year+train_range+val_range+test_range}-01-01"))

In [ ]:
# shared stats (deterministic; identical to nb5's since same source/region/params)
_, SHARED_STATS = mtg.build_standardized_lazy(open_region(ORIGINAL), features, train_year, train_range,
                                              standardize_chl=True)
y_mean, y_std = SHARED_STATS["CHL"][0], SHARED_STATS["CHL"][1]
print("CHL mean/std:", y_mean, y_std)

In [ ]:
# test-year inputs (for the seed-averaged table) + full-record inputs (for the error-over-time plot)
orig_test = open_region(ORIGINAL).sel(time=str(TEST_YEAR))
ds_std_test, _ = mtg.build_standardized_lazy(orig_test, features, train_year, train_range,
                                             standardize_chl=True, stats=SHARED_STATS)
ds_std_test = ds_std_test.load()
_t = ds_std_test.time.values
DATES = pd.to_datetime(_t[np.linspace(0, len(_t) - 1, 10).astype(int)])

ds_std_full, _ = mtg.build_standardized_lazy(open_region(ORIGINAL), features, train_year, train_range,
                                             standardize_chl=True, stats=SHARED_STATS)   # lazy
TRAIN_END = f"{train_year + train_range}-01-01"                 # 2018-01-01 (train = 2015-2017)
VAL_END   = f"{train_year + train_range + val_range}-01-01"     # 2019-01-01
print("eval dates:", [str(d.date()) for d in DATES])

In [ ]:
%%writefile train_seed.py
"""Train one spatial-chunk strategy at one seed, in its own process (fresh GPU each run).
Uses the mindthegap package (mtg.UNet / mtg.make_xbatcher).
Usage: python train_seed.py --strategy <name> --seed <int>"""
import os, argparse, pickle
os.environ.pop("TF_CUDNN_DETERMINISTIC", None)
os.environ.pop("TF_CUDNN_USE_AUTOTUNE", None)
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np, xarray as xr, tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
import mindthegap as mtg


def open_region(path, c):
    ds = xr.open_zarr(path, chunks={})
    if c["SUBSET"] is not None:
        ds = ds.sel(**c["SUBSET"])
    ds = mtg.crop_to_multiple(ds, multiple=8)
    ty, tr, vr, te = c["train_year"], c["train_range"], c["val_range"], c["test_range"]
    return ds.sel(time=slice(f"{ty}-01-01", f"{ty+tr+vr+te}-01-01"))


ap = argparse.ArgumentParser()
ap.add_argument("--strategy", required=True)
ap.add_argument("--seed", type=int, required=True)
ap.add_argument("--config", default="models/spatial_strat/strat_config.pkl")
a = ap.parse_args()
c = pickle.load(open(a.config, "rb"))
S = c["STRATS"][a.strategy]

for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(a.seed)

ds = open_region(c["ORIGINAL"], c)
ty, tr, vr = c["train_year"], c["train_range"], c["val_range"]
LAT, LON = ds.sizes["lat"], ds.sizes["lon"]
ds_std, _ = mtg.build_standardized_lazy(
    ds, c["features"], ty, tr, standardize_chl=True, stats=c["SHARED_STATS"],
    output_chunks={"time": c["DAY_BATCH"], "lat": LAT, "lon": LON})
x_vars = [v for v in ds_std.data_vars if v != "CHL"]
NC = len(x_vars)

dtr = ds_std.sel(time=slice(f"{ty}-01-01", f"{ty+tr}-01-01")).load()
dva = ds_std.sel(time=slice(f"{ty+tr}-01-01", f"{ty+tr+vr}-01-01")).load()
batch = S["batch"]
sig = lambda th, tw: (tf.TensorSpec((th, tw, NC), tf.float32),
                      tf.TensorSpec((th, tw, 1), tf.float32))

if S["sampling"] == "grid":
    th, tw = S["tile"] if S["tile"] else (LAT, LON)
    oh, ow = S["overlap"] if S["overlap"] else (0, 0)
    patch_dims = {"time": c["DAY_BATCH"], "lat": th, "lon": tw}
    overlap = None if (oh, ow) == (0, 0) else {"time": 0, "lat": oh, "lon": ow}
    btr = mtg.make_xbatcher(dtr, patch_dims, overlap=overlap)
    bva = mtg.make_xbatcher(dva, patch_dims, overlap=overlap)
    trds = tf.data.Dataset.from_generator(mtg.make_tf_gen(btr, x_vars), output_signature=sig(th, tw)
        ).shuffle(512, seed=a.seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    vads = tf.data.Dataset.from_generator(mtg.make_tf_gen(bva, x_vars), output_signature=sig(th, tw)
        ).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    steps_tr = (len(btr) * c["DAY_BATCH"]) // batch
    steps_va = (len(bva) * c["DAY_BATCH"]) // batch
else:
    th, tw = S["tile"]; npd = S["n_per_day"]
    def to_arr(dd):
        X = np.stack([np.nan_to_num(dd[v].values, nan=0.0) for v in x_vars], -1).astype(np.float32)
        y = np.nan_to_num(dd["CHL"].values, nan=0.0).astype(np.float32)[..., np.newaxis]
        return X, y
    def crop_gen(Xf, yf, seed):
        rng = np.random.default_rng(seed)
        def g():
            D, H, W, _ = Xf.shape
            for d in range(D):
                for _ in range(npd):
                    yy = int(rng.integers(0, H - th + 1)); xx = int(rng.integers(0, W - tw + 1))
                    yield Xf[d, yy:yy+th, xx:xx+tw], yf[d, yy:yy+th, xx:xx+tw]
        return g
    Xtr, ytr = to_arr(dtr); Xva, yva = to_arr(dva)
    trds = tf.data.Dataset.from_generator(crop_gen(Xtr, ytr, a.seed), output_signature=sig(th, tw)
        ).shuffle(512, seed=a.seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    vads = tf.data.Dataset.from_generator(crop_gen(Xva, yva, a.seed + 1), output_signature=sig(th, tw)
        ).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
    steps_tr = (Xtr.shape[0] * npd) // batch
    steps_va = (Xva.shape[0] * npd) // batch

print(f"[{a.strategy} seed {a.seed}] {S['sampling']} steps {steps_tr}/{steps_va}", flush=True)
model = mtg.UNet((None, None, NC))
es = EarlyStopping(patience=10, restore_best_weights=True)
model.fit(trds, epochs=50, steps_per_epoch=steps_tr,
          validation_data=vads, validation_steps=steps_va, callbacks=[es], verbose=2)
out = f'{c["MODEL_DIR"]}/{a.strategy}_seed{a.seed}.keras'
model.save(out); print("SAVED", out)


In [ ]:
# 5 strategies x 5 seeds = 25 runs, one process each (fresh GPU). Resumable (skip-existing).
import sys, subprocess
cfg = dict(SUBSET=SUBSET, train_year=train_year, train_range=train_range, val_range=val_range,
           test_range=test_range, features=features, DAY_BATCH=DAY_BATCH,
           ORIGINAL=ORIGINAL, RECHUNKED=RECHUNKED, MODEL_DIR=MODEL_DIR,
           SHARED_STATS=SHARED_STATS, STRATS=STRATS)
pickle.dump(cfg, open(f"{MODEL_DIR}/strat_config.pkl", "wb"))

for name in STRATS:
    for s in SEEDS:
        out = f"{MODEL_DIR}/{name}_seed{s}.keras"
        if os.path.exists(out):
            print("skip (exists):", out); continue
        print(f"\n===== {name} seed {s} =====", flush=True)
        rc = subprocess.run([sys.executable, "train_seed.py",
                             "--strategy", name, "--seed", str(s)]).returncode
        print(f"----- {name} seed {s} exited {rc} -----")
print("\nall done")

## Seed-averaged results

Run the two cells below once the sweep above has finished (they auto-skip any seed that is missing).

In [ ]:
# seed-averaged fake-cloud MAE / RMSE per strategy (mean +/- std over seeds)
def fc_scores(model, ds_std, dates):
    xv = [v for v in ds_std.data_vars if v != "CHL"]; ae = se = n = 0.0
    for d in dates:
        sub = ds_std.sel(time=d)
        X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype(np.float32)
        pred = model.predict(X[np.newaxis, ...], verbose=0)[0, :, :, 0] * y_std + y_mean
        truth = sub["CHL"].values * y_std + y_mean
        m = (sub["fake_cloud_flag"].values == 1) & np.isfinite(truth) & np.isfinite(pred)
        ae += np.abs(pred[m]-truth[m]).sum(); se += ((pred[m]-truth[m])**2).sum(); n += m.sum()
    return ae/n, np.sqrt(se/n)

print(f"{'strategy':>14} {'MAE mean':>9} {'MAE std':>8} {'RMSE mean':>10}  (n={len(SEEDS)} seeds)")
for name in STRATS:
    maes, rmses = [], []
    for s in SEEDS:
        p = f"{MODEL_DIR}/{name}_seed{s}.keras"
        if os.path.exists(p):
            mae, rmse = fc_scores(tf.keras.models.load_model(p), ds_std_test, DATES)
            maes.append(mae); rmses.append(rmse)
    if maes:
        print(f"{name:>14} {np.mean(maes):9.4f} {np.std(maes):8.4f} {np.mean(rmses):10.4f}")

In [ ]:
# seed-averaged error over time: one line per strategy (mean of seeds), band = seed range
def compare_perf_seeds(strats, ds_std, seeds, freq="M", block=100):
    xv = [v for v in ds_std.data_vars if v != "CHL"]
    times = pd.to_datetime(ds_std.time.values)
    mdls, daily = {}, {}
    for name in strats:
        for s in seeds:
            p = f"{MODEL_DIR}/{name}_seed{s}.keras"
            if os.path.exists(p):
                mdls[(name, s)] = tf.keras.models.load_model(p)
                daily[(name, s)] = np.full(len(times), np.nan)
    for i in range(0, len(times), block):
        sub = ds_std.isel(time=slice(i, i + block)).load()
        X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype(np.float32)
        truth = sub["CHL"].values * y_std + y_mean
        fake = (sub["fake_cloud_flag"].values == 1) & np.isfinite(truth)
        for key, mdl in mdls.items():
            pred = mdl.predict(X, batch_size=4, verbose=0)[..., 0] * y_std + y_mean
            m = fake & np.isfinite(pred)
            d = np.where(m, np.abs(pred - truth), np.nan)
            with np.errstate(invalid="ignore"):
                daily[key][i:i + d.shape[0]] = np.nanmean(d.reshape(d.shape[0], -1), axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    for name in strats:
        pers = []
        for s in seeds:
            if (name, s) not in daily: continue
            ser = pd.Series(daily[(name, s)], index=times).dropna()
            if freq in ("M", "Y"):
                ser = ser.groupby(ser.index.to_period(freq)).mean(); ser.index = ser.index.to_timestamp()
            pers.append(ser)
        if not pers: continue
        df = pd.concat(pers, axis=1); mean = df.mean(axis=1)
        ax.plot(mean.index, mean.values, lw=1.8, marker="o", ms=3, label=name)
        ax.fill_between(mean.index, df.min(axis=1), df.max(axis=1), alpha=0.12)
    ax.axvspan(pd.Timestamp(f"{train_year}-01-01"), pd.Timestamp(TRAIN_END), color="gray", alpha=0.08)
    ax.axvline(pd.Timestamp(VAL_END), ls="--", c="gray", lw=0.8)
    ax.set_xlabel("time"); ax.set_ylabel("fake-cloud MAE (log Chl-a)")
    ax.set_title(f"per-{freq} gap-fill error by strategy, mean of {len(seeds)} seeds (band = seed range)")
    ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3); plt.show()

compare_perf_seeds(list(STRATS), ds_std_full, SEEDS, freq="M")